In [1]:
#| echo: false
from __future__ import annotations

import os
import re
import subprocess
import sys
from html import escape
from pathlib import Path

from IPython.display import HTML, display


def _lib_dir() -> Path:
    """Locate the katapult package directory (contains pyproject.toml and tests/)."""
    if "QUARTO_PROJECT_DIR" in os.environ:
        d = Path(os.environ["QUARTO_PROJECT_DIR"]).resolve() / "lib"
        if (d / "pyproject.toml").is_file():
            return d
    here = Path.cwd().resolve()
    for ancestor in [here, *here.parents]:
        d = ancestor / "lib"
        if (d / "pyproject.toml").is_file():
            return d
    raise FileNotFoundError(
        "Could not find lib/pyproject.toml. Run Quarto from the repo root or open the notebook from the katapult project."
    )


_ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")


def _strip_ansi(text: str) -> str:
    return _ANSI_RE.sub("", text)


_STATUS_ORDER = (
    "PASSED",
    "FAILED",
    "SKIPPED",
    "ERROR",
    "XFAIL",
    "XPASS",
    "WARNED",
)

_BADGE_STYLE = {
    "PASSED": "background:#198754;color:#fff;",
    "FAILED": "background:#dc3545;color:#fff;",
    "ERROR": "background:#dc3545;color:#fff;",
    "SKIPPED": "background:#fd7e14;color:#fff;",
    "XFAIL": "background:#6c757d;color:#fff;",
    "XPASS": "background:#0dcaf0;color:#000;",
    "WARNED": "background:#ffc107;color:#000;",
}


def _parse_pytest_output(text: str) -> tuple[list[tuple[str, str]], str | None]:
    """Return (list of (nodeid, status), summary line or None)."""
    clean = _strip_ansi(text)
    tests: list[tuple[str, str]] = []
    summary: str | None = None
    for line in clean.splitlines():
        stripped = line.rstrip()
        matched = False
        for status in _STATUS_ORDER:
            token = f" {status}"
            if token not in stripped or "::" not in stripped:
                continue
            try:
                i = stripped.rindex(token)
            except ValueError:
                continue
            nodeid = stripped[:i].strip()
            if "::" not in nodeid:
                continue
            tests.append((nodeid, status))
            matched = True
            break
        if matched:
            continue
        m = re.match(r"^=+\s*(.+?)\s*=+\s*$", stripped)
        if m and any(
            x in m.group(1).lower()
            for x in (
                "passed",
                "failed",
                "error",
                "skipped",
                "deselected",
                "warnings",
                "xfail",
            )
        ):
            summary = m.group(1).strip()
    return tests, summary


def _short_test_name(nodeid: str) -> str:
    return nodeid.split("::", 1)[-1] if "::" in nodeid else nodeid


def _render_pytest_html(
    tests: list[tuple[str, str]],
    summary: str | None,
    full_text: str,
    returncode: int,
) -> str:
    rows: list[str] = []
    for nodeid, status in tests:
        badge = _BADGE_STYLE.get(status, "background:#6c757d;color:#fff;")
        name = escape(_short_test_name(nodeid))
        rows.append(
            f'<div style="margin:0.35em 0;display:flex;align-items:center;gap:0.6em;flex-wrap:wrap;">'
            f'<span style="{badge}padding:2px 8px;border-radius:4px;font-weight:600;font-size:0.75rem;letter-spacing:0.02em;">{status}</span>'
            f'<code style="font-size:0.9em;">{name}</code></div>'
        )

    if not rows:
        body = (
            f'<p style="color:#6c757d;">Could not parse pytest output as individual tests.</p>'
            f'<pre style="background:#f8f9fa;border:1px solid #dee2e6;border-radius:6px;padding:1em;overflow:auto;font-size:0.85em;">'
            f"{escape(full_text.rstrip() or '(no output)')}</pre>"
        )
    else:
        any_fail = any(s in ("FAILED", "ERROR") for _, s in tests)
        ok = returncode == 0 and not any_fail
        bar = "#198754" if ok else "#dc3545"
        sum_html = (
            f'<div style="margin-top:1em;padding:0.6em 0.75em;background:{bar};color:#fff;border-radius:6px;font-weight:600;">'
            f"{escape(summary) if summary else 'Done'}"
            f' <span style="opacity:0.9;font-weight:400;">— exit code {returncode}</span></div>'
        )
        body = (
            f'<div style="font-family:system-ui,-apple-system,sans-serif;line-height:1.5;">'
            f"{''.join(rows)}{sum_html}</div>"
        )

    if tests and (
        returncode != 0 or any(s in ("FAILED", "ERROR") for _, s in tests)
    ):
        body += (
            f'<details style="margin-top:1em;"><summary style="cursor:pointer;font-weight:600;">Full pytest output</summary>'
            f'<pre style="margin-top:0.5em;background:#f8f9fa;border:1px solid #dee2e6;border-radius:6px;padding:1em;overflow:auto;font-size:0.8em;white-space:pre-wrap;">'
            f"{escape(full_text.rstrip() or '(no output)')}</pre></details>"
        )

    return f'<div class="pytest-report" style="margin:1em 0;">{body}</div>'


def run_pytest(*extra_args: str) -> None:
    """Run pytest in lib/ and show a styled summary in the notebook."""
    lib = _lib_dir()
    venv_py = lib / ".venv" / "bin" / "python"
    exe = str(venv_py) if venv_py.is_file() else sys.executable
    cmd = [exe, "-m", "pytest", "tests/", "-v", "--tb=short", *extra_args]
    result = subprocess.run(cmd, cwd=str(lib), capture_output=True, text=True)
    out = (result.stdout or "") + (result.stderr or "")
    if not out.strip():
        out = "(no output)"
    full_clean = _strip_ansi(out)
    tests, summary = _parse_pytest_output(out)
    html = _render_pytest_html(tests, summary, full_clean, result.returncode)
    display(HTML(html))

# Katapult library unit tests

This notebook documents the pytest suite under `lib/tests/` and runs each group so you can see live results when you execute the cells (or when Quarto renders this page).

**Source:** [`lib/tests/test_commands.py`](../../lib/tests/test_commands.py) — helpers in [`katapult.commands`](../../lib/src/katapult/commands.py) and the `kat` CLI (`init`, `hub`, `config`, and the internal `rich` demo command).


## Overview

Tests fall into three layers: pure path/JSON helpers used by `kat init`, Click CLI exercises with mocks (no real Docker or cookiecutter prompts), and optional grouping by pytest `-k` expression.

```mermaid
flowchart TB
  subgraph helpers [Helper functions]
    merge["_merge_overrides"]
    cwr["_apply_copy_without_render"]
    otc["_override_template_has_content"]
  end
  subgraph cli [CLI commands]
    init["init"]
    hub["hub"]
    cfg["config"]
    rich["rich"]
  end
  merge --> init
  cwr --> init
  otc --> init
```


## `_merge_overrides`

When `kat init` applies `~/.katapult/template/` overrides, files are copied into the cookiecutter project directory inside a temporary merged template, preserving relative paths.

- **`test_merge_overrides_copies_nested_files`** — nested directories and files land under `dst` with the same structure.
- **`test_merge_overrides_skips_directories`** — empty directories under the source do not create files under `dst` (only files are copied).

```mermaid
flowchart LR
  src["override src"] --> merge["_merge_overrides"]
  merge --> dst["dst = .../ project_slug /"]
```


In [2]:
run_pytest("-k", "merge_overrides")

## `_apply_copy_without_render`

Merges glob patterns from `~/.katapult/copy_without_render` into the template's `cookiecutter.json` field `_copy_without_render` so those paths are copied without Jinja templating.

| Test | What it checks |
|------|----------------|
| `merges_patterns` | Lines from the ignore file append to existing list; `#` comments and blanks skipped |
| `deduplicates` | Duplicate patterns appear once |
| `non_list_existing_reset` | If existing value is not a list, it is replaced |
| `missing_ignore_noop` | No ignore file → no change to JSON |
| `empty_patterns_noop` | Only comments/empty lines → no change |
| `invalid_json_raises` | Bad `cookiecutter.json` → `JSONDecodeError` |
| `missing_cookiecutter_raises` | Missing JSON path → `FileNotFoundError` |


In [3]:
run_pytest("-k", "apply_copy_without_render")

## `_override_template_has_content`

Decides whether `kat init` should use the merge-and-temp-template path: **True** only if the override directory exists and contains at least one file (any depth).

- Missing path → False  
- Empty directory → False  
- File at root or nested → True


In [4]:
run_pytest("-k", "override_template_has_content")

## CLI: `rich` and `config`

- **`rich`** — demo command (not registered on the main `kat` group); prints a Rich table; exit code 0.
- **`config`** — appends PATH-augmentation block to `~/.bashrc` once; second run reports already present. Tests patch `Path.home` to a temporary directory.

```mermaid
sequenceDiagram
  participant U as User
  participant K as kat config
  participant B as .bashrc
  U->>K: invoke
  K->>B: append marker block if missing
  K-->>U: message
```


In [5]:
run_pytest("-k", "rich or config")

## CLI: `kat init`

Cookiecutter is **mocked** so tests never prompt interactively.

- **`--no-overrides`** — calls `cookiecutter` with the built-in template directory only.
- **With overrides** — copies template to a temp dir, merges `~/.katapult/template/` into `{{cookiecutter.project_slug}}/`, then calls `cookiecutter` on the merged tree. One test asserts merged files inside a `cookiecutter` side-effect while the temp dir still exists.
- **`--no-overrides` with template files present** — still uses the stock template path (skips merge).


In [6]:
run_pytest("-k", "init")

## CLI: `kat hub`

Docker is **fully mocked** (`docker.from_env`). Tests drive confirms with `CliRunner` input:

- Traefik already running → no `create` / `run`.
- No network → user confirms → `networks.create` and `containers.run`.
- User declines network or Traefik launch → abort messages, no side effects.


In [7]:
run_pytest("-k", "hub")

## Full test suite

Runs all tests in `lib/tests/` (same as `cd lib && uv run pytest tests/ -v`).


In [8]:
run_pytest()